In [1]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path(r"G:\Projects\Dementia_Detection_Model")

baseline_report_path = (
    PROJECT_ROOT / "outputs" / "evaluation" /
    "baseline" / "classification_report.csv"
)

cbam_report_path = (
    PROJECT_ROOT / "outputs" / "evaluation" /
    "cbam" / "classification_report.csv"
)

baseline_report = pd.read_csv(baseline_report_path)
cbam_report = pd.read_csv(cbam_report_path)

print("BASELINE")
display(baseline_report)

print("\nCBAM")
display(cbam_report)

BASELINE


,Unnamed: 0,precision,recall,f1-score,support
0,MildDemented,0.605453,0.75500,0.672007,1000.00000
1,ModerateDemented,0.888393,0.99500,0.938679,1000.00000
2,NonDemented,0.618529,0.68100,0.648263,1000.00000
3,VeryMildDemented,0.631579,0.33600,0.438642,1000.00000
4,accuracy,0.691750,0.69175,0.691750,0.69175
5,macro avg,0.685988,0.69175,0.674398,4000.00000
6,weighted avg,0.685988,0.69175,0.674398,4000.00000
7,test_loss,0.762870,0.76287,0.762870,0.76287
8,test_accuracy,0.691750,0.69175,0.691750,0.69175



CBAM


,Unnamed: 0,precision,recall,f1-score,support
0,MildDemented,0.888779,0.895000,0.891878,1000.000000
1,ModerateDemented,0.991080,1.000000,0.995520,1000.000000
2,NonDemented,0.835891,0.708000,0.766649,1000.000000
3,VeryMildDemented,0.724714,0.824000,0.771175,1000.000000
4,accuracy,0.856750,0.856750,0.856750,0.856750
5,macro avg,0.860116,0.856750,0.856305,4000.000000
6,weighted avg,0.860116,0.856750,0.856305,4000.000000
7,test_loss,0.346303,0.346303,0.346303,0.346303
8,test_accuracy,0.856750,0.856750,0.856750,0.856750


In [3]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import precision_recall_fscore_support

# ============================================================
# PATHS
# ============================================================

PROJECT_ROOT = Path(r"G:\Projects\Dementia_Detection_Model")

EVAL_DIR = PROJECT_ROOT / "outputs" / "evaluation"
EVAL_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# CLASS ORDER
# ============================================================

class_names = [
    "NonDemented",
    "VeryMildDemented",
    "MildDemented",
    "ModerateDemented"
]

# ============================================================
# ORIGINAL EFFICIENTNET CONFUSION MATRIX
# Original order:
# Mild, Moderate, NonDemented, VeryMild
# ============================================================

baseline_old = np.array([
    [755, 39, 122, 84],
    [2, 995, 1, 2],
    [173, 36, 681, 110],
    [317, 50, 297, 336]
])

# Reorder to:
# NonDemented, VeryMildDemented, MildDemented, ModerateDemented

order = [2, 3, 0, 1]

baseline_cm = baseline_old[np.ix_(order, order)]


# ============================================================
# ORIGINAL CBAM CONFUSION MATRIX
# Original order:
# Mild, Moderate, NonDemented, VeryMild
# ============================================================

cbam_old = np.array([
    [895, 1, 22, 82],
    [0, 1000, 0, 0],
    [57, 4, 708, 231],
    [55, 4, 117, 824]
])

cbam_cm = cbam_old[np.ix_(order, order)]


# ============================================================
# FINE-TUNED CBAM CONFUSION MATRIX
# Already in correct order
# ============================================================

finetuned_cm = np.array([
    [973, 0, 5, 22],
    [0, 1000, 0, 0],
    [54, 1, 772, 173],
    [52, 2, 98, 848]
])


# ============================================================
# FUNCTION TO CALCULATE METRICS
# ============================================================

def metrics_from_cm(cm, class_names):

    y_true = []
    y_pred = []

    for actual in range(len(cm)):
        for predicted in range(len(cm)):
            count = cm[actual, predicted]

            y_true.extend([actual] * count)
            y_pred.extend([predicted] * count)

    precision, recall, f1, support = (
        precision_recall_fscore_support(
            y_true,
            y_pred,
            labels=range(len(class_names)),
            zero_division=0
        )
    )

    accuracy = np.trace(cm) / np.sum(cm)

    report = pd.DataFrame({
        "Class": class_names,
        "Precision": precision,
        "Recall": recall,
        "F1-Score": f1,
        "Support": support
    })

    macro_precision = precision.mean()
    macro_recall = recall.mean()
    macro_f1 = f1.mean()

    return report, accuracy, macro_precision, macro_recall, macro_f1


# ============================================================
# CALCULATE ALL THREE
# ============================================================

baseline_report, baseline_acc, baseline_precision, baseline_recall, baseline_f1 = \
    metrics_from_cm(baseline_cm, class_names)

cbam_report, cbam_acc, cbam_precision, cbam_recall, cbam_f1 = \
    metrics_from_cm(cbam_cm, class_names)

finetuned_report, finetuned_acc, finetuned_precision, finetuned_recall, finetuned_f1 = \
    metrics_from_cm(finetuned_cm, class_names)


# ============================================================
# PRINT RESULTS
# ============================================================

print("\n========== BASELINE ==========")
print(baseline_report.round(4).to_string(index=False))

print("\n========== CBAM ==========")
print(cbam_report.round(4).to_string(index=False))

print("\n========== FINE-TUNED CBAM ==========")
print(finetuned_report.round(4).to_string(index=False))


========== BASELINE ==========
           Class  Precision  Recall  F1-Score  Support
     NonDemented     0.6185   0.681    0.6483     1000
VeryMildDemented     0.6316   0.336    0.4386     1000
    MildDemented     0.6055   0.755    0.6720     1000
ModerateDemented     0.8884   0.995    0.9387     1000

========== CBAM ==========
           Class  Precision  Recall  F1-Score  Support
     NonDemented     0.8359   0.708    0.7666     1000
VeryMildDemented     0.7247   0.824    0.7712     1000
    MildDemented     0.8888   0.895    0.8919     1000
ModerateDemented     0.9911   1.000    0.9955     1000

========== FINE-TUNED CBAM ==========
           Class  Precision  Recall  F1-Score  Support
     NonDemented     0.9018   0.973    0.9360     1000
VeryMildDemented     0.9970   1.000    0.9985     1000
    MildDemented     0.8823   0.772    0.8235     1000
ModerateDemented     0.8130   0.848    0.8302     1000


In [4]:
comparison = pd.DataFrame({
    "Metric": [
        "Test Loss",
        "Accuracy",
        "Macro Precision",
        "Macro Recall",
        "Macro F1"
    ],

    "EfficientNetV2B0": [
        0.76287,
        baseline_acc,
        baseline_precision,
        baseline_recall,
        baseline_f1
    ],

    "EfficientNetV2B0 + CBAM": [
        0.346303,
        cbam_acc,
        cbam_precision,
        cbam_recall,
        cbam_f1
    ],

    "Fine-Tuned + CBAM": [
        0.2524,
        finetuned_acc,
        finetuned_precision,
        finetuned_recall,
        finetuned_f1
    ]
})

print(comparison.round(4).to_string(index=False))

         Metric  EfficientNetV2B0  EfficientNetV2B0 + CBAM  Fine-Tuned + CBAM
      Test Loss            0.7629                   0.3463             0.2524
       Accuracy            0.6918                   0.8568             0.8982
Macro Precision            0.6860                   0.8601             0.8985
   Macro Recall            0.6918                   0.8568             0.8982
       Macro F1            0.6744                   0.8563             0.8970


In [5]:
# Save overall comparison
comparison.to_csv(
    EVAL_DIR / "three_model_comparison.csv",
    index=False
)

# Save individual classification reports
baseline_report.to_csv(
    EVAL_DIR / "baseline_classification_report.csv",
    index=False
)

cbam_report.to_csv(
    EVAL_DIR / "cbam_classification_report.csv",
    index=False
)

finetuned_report.to_csv(
    EVAL_DIR / "finetuned_classification_report.csv",
    index=False
)

# Save confusion matrices
pd.DataFrame(
    baseline_cm,
    index=class_names,
    columns=class_names
).to_csv(
    EVAL_DIR / "baseline_confusion_matrix.csv"
)

pd.DataFrame(
    cbam_cm,
    index=class_names,
    columns=class_names
).to_csv(
    EVAL_DIR / "cbam_confusion_matrix.csv"
)

pd.DataFrame(
    finetuned_cm,
    index=class_names,
    columns=class_names
).to_csv(
    EVAL_DIR / "finetuned_confusion_matrix.csv"
)

print("All evaluation files saved successfully!")
print(f"Location: {EVAL_DIR}")

All evaluation files saved successfully!
Location: G:\Projects\Dementia_Detection_Model\outputs\evaluation
